# Module: Reranking in RAG (Cohere, BGE-Reranker, Cross-Encoders)

Even with advanced hybrid search, initial retrieval often fetches a broad pool of candidates (e.g., top 20 or 50 chunks) containing noise, surface-level keyword hits, or irrelevant context. Reranking acts as a precision filter placed right before the LLM, scoring and re-ordering candidates to ensure only the absolute highest-quality context makes it into the prompt.

## 1. Bi-Encoders vs. Cross-Encoders (The Core Concept)
To understand why reranking works, you need to understand the architecture difference between standard retrieval and reranking:

### Bi-Encoders (Used in Retrieval/Embeddings):

The query and the document chunk are processed independently through an embedding model.

Their vector representations are pre-calculated and stored. Similarity is computed via quick geometry (cosine distance).

**Analogy:** Reading a book summary and a question separately, then guessing if they match without cross-examining them line-by-line.

### Cross-Encoders (Used in Reranking):

The query and the document chunk are fed together as a single input string into a transformer model (e.g., [CLS] Query [SEP] Document Chunk [SEP]).

The model performs full self-attention across both texts simultaneously, capturing deep semantic nuances, negations, and fine-grained logic.

**Analogy:** Having a person read both the question and the document side-by-side to grade relevance explicitly.

The Trade-off: Cross-encoders are far too computationally intensive to run across millions of database vectors, but they are exceptionally accurate when applied to a narrow shortlist (e.g., the top 20 chunks returned by initial retrieval).

## 2. Industry Standard Reranking Solutions

A. Cohere Rerank (Managed API)Cohere's Rerank endpoint is widely considered the gold standard for production-grade enterprise RAG.

Pros: State-of-the-art ranking performance, zero local hardware or GPU management, natively handles multilingual text (100+ languages), semi-structured formats (JSON, code, tables), and generous context windows.  

Cons: Paid commercial API with per-request pricing; requires sending text data to an external provider.Python Example:

In [ ]:
import cohere

co = cohere.Client("YOUR_API_KEY")

response = co.rerank(
    model="rerank-v3.5",
    query="What is the company policy on remote work equipment reimbursement?",
    documents=[
        "Employees can claim up to $500 for home office setups.",
        "The cafeteria serves lunch from 12 PM to 2 PM daily.",
        "Travel expenses are reimbursed via the Concur system."
    ],
    top_n=2
)

for idx, result in enumerate(response.results):
    print(f"Rank {idx+1} (Score: {result.relevance_score:.4f}): {result.document['text']}")

## B. BGE-Reranker (Open-Source / Local)

Developed by the Beijing Academy of Artificial Intelligence (BAAI), the bge-reranker family (such as bge-reranker-v2-m3) provides powerful open-weights cross-encoders.  

Pros: 100% open-source and free to run locally, eliminating data privacy boundaries. Highly competitive accuracy benchmarks against proprietary models.

Cons: Requires local compute resources (ideally a GPU with sufficient VRAM for real-time latency demands).

Python Example (using Hugging Face / FlagEmbedding):

In [ ]:
from FlagEmbedding import FlagReranker

# Load open-source cross-encoder model locally
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

query = "What is the policy on remote work equipment?"
passages = [
    "Employees can claim up to $500 for home office setups.",
    "The cafeteria serves lunch from 12 PM to 2 PM daily."
]

# Compute cross-encoder relevance scores
scores = reranker.compute_score([[query, p] for p in passages])
print(scores)  # Higher score = higher relevance

## 3. The Modern RAG Pipeline Architecture with Reranking
When you assemble all components from Loading up to Reranking, a state-of-the-art production query flow looks like this:

In [ ]:
[ User Query ]
       │
       ├─────────────────────────┐
       ▼                         ▼
[ Dense Retriever ]     [ Sparse Retriever (BM25) ]
       │                         │
       └───────────┬─────────────┘
                   ▼
       [ Hybrid Fusion (RRF) ] ──> (Top 30 Chunks)
                   │
                   ▼
        [ Cross-Encoder Reranker ] (Cohere / BGE)
                   │
                   ▼
           (Top 3-5 Chunks) ────> [ LLM Context Window ]

### Quick Comparison Table

| Feature | Bi-Encoder (Standard Retrieval) | Cross-Encoder (Reranker)
| :--- | :--- | :--- |
| Execution Phase | Stage 1: Retrieval (Searches millions of docs) | Stage 2: Filtering (Refines top 20–50 candidates)
| Speed / Latency | Ultra-fast (Milliseconds via pre-computed index) | Slower (Computes dynamic query-document interactions)
| Accuracy | Good for broad semantic filtering | Exceptional for precise ranking and subtle reasoning
| Infrastructure | Vector database index (HNSW/IVFFlat) | Dedicated transformer scoring node or API endpoint